# 03 · Change over time: growth models

**The question:** does job satisfaction change across survey waves, and do people change differently from one another?

Culture Amp customers survey the same people every cycle, so their data has a second kind of nesting: **survey responses nested in people**, and people nested in teams. Your European soccer data has the same shape, with ratings nested in players.

**The data:** `univbct`, from Bliese & Ployhart (2002). It follows 495 soldiers in 8 companies at three time points, measuring job satisfaction (`JSAT`), organizational commitment (`COMMIT`) and readiness (`READY`).

**By the end you'll be able to:**
1. Move repeated-measures data between wide and long form
2. Explain why ignoring repeated measures can make a standard error *too large*, the opposite of notebook 01
3. Fit and interpret a growth model with random intercepts and slopes
4. Separate within-person change from between-person differences
5. Spot regression to the mean before it fools you or a customer
6. Know when there are too few groups to model as a random effect

In [ ]:
# colab: runtime > change runtime type > R. the first run installs packages (about a minute)
for (pkg in c("multilevel", "ggplot2")) {
  if (!requireNamespace(pkg, quietly = TRUE)) install.packages(pkg)
}
suppressPackageStartupMessages({
  library(multilevel)  # also loads nlme, which fits the models
  library(ggplot2)
})
options(digits = 3, repr.plot.width = 9, repr.plot.height = 4.5)
ctl <- lmeControl(opt = "optim")  # a sturdier optimizer for models with random slopes

# checks your answers in the "your turn" cells
check <- function(yours, target, label = "answer", tol = 0.005) {
  if (is.null(yours)) return(message("[todo] ", label, ": not filled in yet"))
  keep <- !is.na(target)
  gap <- abs(as.numeric(yours)[keep] - as.numeric(target)[keep])
  if (length(yours) == length(target) && !anyNA(gap) && max(gap) <= tol) {
    message("[ok] ", label, ": ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "))
  } else {
    message("[not yet] ", label, ": yours = ", paste(round(head(as.numeric(yours), 3), 3), collapse = ", "),
            ". recheck the formula")
  }
}

data(univbct)
head(univbct[, c("SUBNUM", "COMPANY", "TIME", "JSAT", "COMMIT", "READY", "JOBSAT1", "JOBSAT2", "JOBSAT3")], 6)

## Part A · Wide vs. long

The data arrives in both shapes. `JOBSAT1`–`JOBSAT3` are **wide** (one column per wave, repeated on every row for that soldier). `TIME` and `JSAT` are **long** (one row per soldier per wave). Growth models need long data, and in real work you usually have to build it yourself.

In [ ]:
# one row per soldier
wide <- univbct[univbct$TIME == 0, c("SUBNUM", "COMPANY", "JOBSAT1", "JOBSAT2", "JOBSAT3")]
cat("wide:", nrow(wide), "rows\n")

# your turn: reshape `wide` to long form with columns SUBNUM, COMPANY, TIME (0, 1, 2) and jobSat
# hint: base R's reshape() with direction = "long". read ?reshape for varying, v.names, timevar, times, idvar
long <- NULL

if (is.null(long)) {
  message("[todo] long form: not filled in yet")
} else {
  long <- long[order(long$SUBNUM, long$TIME), ]
  ref <- univbct[order(univbct$SUBNUM, univbct$TIME), ]
  check(nrow(long), nrow(ref), "row count", tol = 0)
  check(long$jobSat, ref$JSAT, "values match JSAT")
}

## Part B · Look before you model

In [ ]:
# wave averages (na.rm because some soldiers skipped waves)
round(sapply(c("JSAT", "COMMIT", "READY"), function(v) tapply(univbct[[v]], univbct$TIME, mean, na.rm = TRUE)), 3)

set.seed(7)
somePeople <- sample(unique(univbct$SUBNUM), 40)
answered <- subset(univbct, !is.na(JSAT))
ggplot(subset(answered, SUBNUM %in% somePeople), aes(TIME, JSAT, group = SUBNUM)) +
  geom_line(alpha = 0.35) +
  stat_summary(data = answered, aes(group = 1), fun = mean, geom = "line", colour = "steelblue", linewidth = 1.3) +
  scale_x_continuous(breaks = 0:2) +
  labs(x = "wave", y = "job satisfaction", title = "40 soldiers' trajectories (grey) and the overall average (blue)")

The average barely moves. The individual lines go everywhere: some people rise, some fall, and they start at very different levels. A growth model is built to describe exactly that: the average trajectory **and** how people vary around it.

## Part C · Ignore the repeated measures, and the standard error goes... which way?

**Predict first:** in notebook 01, ignoring nesting made the standard error of a team-level predictor **too small**. Here the predictor is `TIME`, which changes *within* each person. If you ignore that the same soldiers appear three times, will the standard error of the time slope be too small or too large?

In [ ]:
olsTime <- lm(JSAT ~ TIME, data = univbct)
riTime <- lme(JSAT ~ TIME, random = ~ 1 | SUBNUM, data = univbct, na.action = na.omit)
rsTime <- lme(JSAT ~ TIME, random = ~ TIME | SUBNUM, data = univbct, na.action = na.omit, control = ctl)

grab <- function(fit) {
  if (inherits(fit, "lm")) return(unname(coef(summary(fit))["TIME", c("Estimate", "Std. Error", "Pr(>|t|)")]))
  unname(summary(fit)$tTable["TIME", c("Value", "Std.Error", "p-value")])
}
timeResults <- rbind(
  "ordinary regression (rows treated as independent)" = grab(olsTime),
  "random intercept per soldier"                      = grab(riTime),
  "random intercept and slope per soldier"            = grab(rsTime)
)
colnames(timeResults) <- c("slope", "stdError", "p")
round(timeResults, 4)

The standard error goes **down** once each soldier has their own intercept, and the slope crosses the .05 line.

Why: soldiers differ a lot in their overall level of satisfaction. The ordinary regression treats that stable person-to-person difference as noise around the time trend. The multilevel model sets it aside as "who you are," so the change *within* each person is measured against much less noise. This is the same logic as a paired t-test beating an independent-samples t-test.

**The full rule:**
- Ignoring nesting makes standard errors for **higher-level** predictors (team climate, a person's stable traits) **too small**
- It often makes them **too large** for **lower-level** predictors that vary within the group (time, the occasion-to-occasion swings)

Either way the ordinary regression is wrong, and you can't tell in advance which way.

## Part D · The growth model, unpacked

Two levels, written separately:

$$\textbf{Level 1 (waves within a soldier):}\quad \text{JSAT}_{ti} = \pi_{0i} + \pi_{1i}\,\text{TIME}_{ti} + e_{ti}$$
$$\textbf{Level 2 (soldiers):}\quad \pi_{0i} = \beta_{00} + r_{0i}, \qquad \pi_{1i} = \beta_{10} + r_{1i}$$

Every soldier gets their own starting level $\pi_{0i}$ and their own rate of change $\pi_{1i}$. The model estimates the averages ($\beta_{00}$, $\beta_{10}$), how much soldiers vary around them (the variances of $r_{0i}$ and $r_{1i}$), and whether starting level and change are related (their correlation).

In [ ]:
anova(riTime, rsTime)   # do soldiers really differ in their rate of change?
VarCorr(rsTime)

In [ ]:
nullTime <- lme(JSAT ~ 1, random = ~ 1 | SUBNUM, data = univbct, na.action = na.omit)
VarCorr(nullTime)

# your turn: from the null model above, what share of the variance in job satisfaction
# is stable differences between soldiers? (same formula as ICC(1) in notebook 01)
personIcc <- NULL

check(personIcc, {v <- as.numeric(VarCorr(nullTime)[, "Variance"]); v[1] / sum(v)}, "person-level ICC")

About half the variance in any one response is the stable "who you are" part. That is far larger than the 4% team share in notebook 01. Repeated responses from one person resemble each other much more than responses from teammates do.

Look at the intercept–slope **correlation** in `VarCorr(rsTime)` (about −0.58). Soldiers who start lower tend to rise more. Before you call that "the least satisfied improve the most," read Part F.

## Part E · Within-person change vs. between-person differences

Commitment is measured at every wave too, so it can be split exactly the way you split leadership in notebook 02:

- **Between-person:** are soldiers who are *generally* more committed also more satisfied?
- **Within-person:** in a wave where a soldier is *more committed than usual for them*, are they also more satisfied than usual?

The within-person version controls for everything stable about a person (personality, background, rank), because each soldier is compared only with themselves. That makes it the closest a survey gets to "change in X goes with change in Y."

In [ ]:
# your turn: person-mean center COMMIT
# personCommit = each soldier's average commitment across the waves they answered (watch the missing values)
# withinCommit = COMMIT minus personCommit
# hint: ave() takes a FUN argument, and mean() needs na.rm = TRUE here
personCommit <- NULL
withinCommit <- NULL

refPerson <- tapply(univbct$COMMIT, univbct$SUBNUM, mean, na.rm = TRUE)[as.character(univbct$SUBNUM)]
check(personCommit, refPerson, "person mean of COMMIT")
check(withinCommit, univbct$COMMIT - refPerson, "within-person COMMIT")

In [ ]:
# the rest of the notebook uses these two columns (they match your answers if your checks passed)
univbct$personCommit <- ave(univbct$COMMIT, univbct$SUBNUM, FUN = function(x) mean(x, na.rm = TRUE))
univbct$withinCommit <- univbct$COMMIT - univbct$personCommit

withinPerson <- lme(JSAT ~ TIME + withinCommit + personCommit, random = ~ TIME | SUBNUM,
                    data = univbct, na.action = na.omit, control = ctl)
round(summary(withinPerson)$tTable, 4)

**Your interpretation.** Write one sentence for each coefficient: `withinCommit`, then `personCommit`. Then answer:

- Which of the two is closer to the question a manager cares about ("if this person's commitment goes up, will their satisfaction?"), and why?
- What does the within-person estimate still *not* rule out? (Hint: something that changes over time and could push both up in the same wave.)

*Your interpretation:*

## Part F · Does where you start predict how you change?

Does a soldier's commitment at wave 1 predict how their job satisfaction changes afterward? In R, that's a `TIME × starting commitment` interaction. Centering starting commitment makes the `TIME` coefficient mean "change for someone with average starting commitment."

In [ ]:
startAvg <- mean(univbct$COMMIT1[univbct$TIME == 0], na.rm = TRUE)
univbct$commitStart <- univbct$COMMIT1 - startAvg

startModel <- lme(JSAT ~ TIME * commitStart, random = ~ TIME | SUBNUM,
                  data = univbct, na.action = na.omit, control = ctl)
round(summary(startModel)$tTable, 4)

In [ ]:
startSd <- sd(univbct$commitStart[univbct$TIME == 0], na.rm = TRUE)
grid <- expand.grid(TIME = 0:2, commitStart = c(-1, 1) * startSd)
grid$predicted <- predict(startModel, newdata = grid, level = 0)
grid$group <- ifelse(grid$commitStart < 0, "low starting commitment (-1 SD)", "high starting commitment (+1 SD)")

ggplot(grid, aes(TIME, predicted, colour = group)) +
  geom_line(linewidth = 1) + geom_point() +
  scale_x_continuous(breaks = 0:2) +
  labs(x = "wave", y = "predicted job satisfaction", colour = NULL,
       title = "Predicted satisfaction by starting commitment")

The high-commitment group starts much more satisfied, then the two lines **converge**. Before you tell a customer "your most committed people are losing satisfaction," consider **regression to the mean**.

Any single survey score is part true level and part noise. Soldiers who score unusually high at wave 1 are, on average, partly there by luck (a good week, a generous mood). The next measurement drifts back toward their true level, so they appear to decline even if nothing real changed. The intercept–slope correlation in Part D carries the same warning.

**Your turn: write it out.**
- In 2–3 sentences, explain regression to the mean to a non-technical HR leader.
- Name one way you could check whether the convergence here is real. Ideas to consider: does it appear when you group people by *wave 2* commitment instead? Does a measure with less noise (an average of several items, or a latent variable) show the same thing? Is there a comparison group?

*Your answer:*

## Part G · What about the 8 companies?

Soldiers are also nested in companies. Should company get a random effect too?

In [ ]:
round(tapply(univbct$JSAT, univbct$COMPANY, mean, na.rm = TRUE), 2)

threeLevel <- lme(JSAT ~ TIME, random = ~ 1 | COMPANY/SUBNUM, data = univbct, na.action = na.omit)
VarCorr(threeLevel)

The company averages clearly differ. A random effect for company, though, asks the model to estimate a *variance* from just 8 values, and a variance estimated from 8 numbers is very unstable. Common advice is to have at least 20–50 groups before treating a level as random, especially if you care about that level's variance or its predictors.

With few groups, use **fixed effects** instead: one indicator per company. That controls for every stable company difference without pretending to estimate a distribution of companies. You can then compare models with and without company. Their fixed effects differ, so both are refit with ML first (see notebook 02).

In [ ]:
companyFixed <- lme(JSAT ~ TIME + COMPANY, random = ~ TIME | SUBNUM,
                    data = univbct, na.action = na.omit, control = ctl)
anova(update(rsTime, method = "ML"), update(companyFixed, method = "ML"))

The customer version: an organization with 8 departments has 8 departments, not a sample from a population of departments. Control for them. A customer with 300 teams is different, and team is worth a random effect, as in notebook 01.

## Part H · Missing waves

In [ ]:
wavesAnswered <- tapply(!is.na(univbct$JSAT), univbct$SUBNUM, sum)
table(wavesAnswered)
cat("observations the growth model used:", nobs(rsTime), "of", nrow(univbct), "\n")

A repeated-measures ANOVA would keep only the 421 soldiers who answered all three waves. The growth model keeps every response, so about 70 soldiers who missed a wave still count.

The assumption that makes this valid is **missing at random**: whether someone skipped a wave can depend on things you've measured (such as their earlier answers), but not on the answer they would have given. Survey non-response is often *not* random. People who are disengaged are less likely to answer an engagement survey. Be ready to say that out loud in an interview.

## Part I · Apply it to your own data

Your European soccer dataset has exactly this structure: ratings (waves) nested in players (people), and players nested in teams. Rebuild this notebook on it:

1. Reshape to long form: one row per player per rating date or season, with a time variable that starts at 0
2. Fit the null model and get the player-level ICC
3. Fit random-intercept and random-slope growth models, then compare them
4. Add a time-invariant predictor (age at the first rating is a natural one) and its interaction with time
5. Try a within-player vs. between-player split on a time-varying attribute
6. Compare the standard error of the time slope with an ordinary regression's

One complication: players change teams, so players aren't neatly nested inside one team. That's a *cross-classified* structure. Leave teams out at first, and note it as a limitation.

## Wrap-up

In your own words, two sentences each:

1. Why the standard error went *down* here but *up* in notebook 01
2. What a random slope for time tells a customer that the average trend doesn't
3. Why you'd use fixed effects for 8 companies but a random effect for 300 teams
4. What regression to the mean is, and where it showed up in this notebook

*Your answers:*

1.

2.

3.

4.